# Quantitative localization from saved ALBEF heatmaps

**Purpose:** assess A0 versus A1, A2, the three A3 coefficients and A4 on existing
VinDr test heatmaps. No training, inference, classification re-evaluation or thesis
editing occurs. External models and multi-branch fusion are deliberately excluded
from this first, within-attribution comparison.

Run cells in order on the server. Requires Python 3.10+, NumPy, pandas, SciPy,
PyTorch, Matplotlib and Pillow (for optional existing-image previews).
The analysis functions are embedded below: **this notebook is standalone**.
The identical editable module is `analysis/localization_pointing_game.py`.

This is a retrospective evaluation of existing checkpoints, not a preregistered
study. Freezing these choices does not undo prior inspection of selected heatmaps.

## Freeze the protocol before inspecting quantitative outcomes

- **Cohort:** every image/finding pair with a valid target-pathology box in the
  specified test annotation CSV and dimensions in `test_meta.csv`. Each image
  contributes once per pathology, regardless of box count. This is a
  **box-annotated-positive cohort**, not the whole 3,000-image test denominator.
  It does not measure false-positive localization on negative images.
- **Primary endpoint:** pointing-game hit: one saved-grid global-maximum point
  inside **any** target box. Box boundaries are included. Do not threshold the CAM
  or optimize an extraction layer against these test annotations.
- **Geometry:** `dim1 = original width`, `dim0 = original height`. A saved pixel
  `(row, col)` maps to original coordinates `((col+.5)*width/map_width,
  (row+.5)*height/map_height)`. No interpolation, smoothing or intensity
  renormalization is used for scoring. This convention assumes a full-image
  rectangular resize without crop, flip or padding. **If this is untrue, stop and
  supply the actual inverse transform; changing dimensions alone is insufficient.**
- **Ties:** uniformly select an exact global maximum using a stable image/finding
  seed. Report the tie count, expected hit over all tied maxima and any-/all-peak
  bounds as sensitivity diagnostics. Zero and constant maps are primary misses
  because they supply no spatial discrimination; missing/invalid maps are not
  silently counted as misses or dropped.
- **Coverage:** default is strict: every eligible pair must have a valid map for
  every configured method. Incomplete coverage writes diagnostics and stops.
  Fix paths or regenerate missing *saved outputs separately*, rather than selecting
  images according to hit outcomes. Explicitly enabling incomplete mode uses one
  common complete-case intersection across **all methods per pathology**, records
  exclusions and labels the results exploratory. If only a selected gallery was
  extracted, a complete-case analysis does not become a representative test study.
- **Uncertainty:** per-method 95% Wilson intervals; paired image-bootstrap 95%
  percentile intervals for candidate-minus-A0 hit-rate differences; exact
  two-sided McNemar tests. Holm correction covers all configured non-A0 comparisons
  across both findings (12 by default). Intervals are pointwise, not simultaneous.
  Image resampling assumes independent images. If several images share a patient,
  patient identifiers and cluster-aware inference are needed instead. These
  intervals do not estimate variability across training seeds.
- **Interpretation:** a peak hit measures box alignment, not lesion extent,
  clinical usefulness or disease-specific causal reasoning. Large boxes make hits
  easier. Tie policy and per-image normalization affect interpretation. Do not
  select a new checkpoint or claim statistical superiority from raw unadjusted
  p-values. A0 and all guidance maps must use the same confirmed ITC-margin,
  penultimate-layer extraction and best-validation-AUC checkpoint policy.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Paths from the researcher's confirmed configuration, NOT the historical
# notebook's outdated best_cardio path. Verify every path on your server.
BASE = Path('/home/vault/iwi5/iwi5362h/results')
SOURCE_DIRS = {
    'A0': BASE / 'heatmaps_itc_margin/original',
    'A1': BASE / 'anatomy_itc_margin_layer10/A1',
    'A2': BASE / 'anatomy_itc_margin_layer10/A2',
    'A3_lambda_0.01': BASE / 'anatomy_itc_margin_layer10/A3_lambda_0.01',
    'A3_lambda_0.005': BASE / 'anatomy_itc_margin_layer10/A3_lambda_0.005',
    'A3_lambda_0.001': BASE / 'anatomy_itc_margin_layer10/A3_lambda_0.001',
    'A4': BASE / 'anatomy_itc_margin_layer10/A4',
}
ANNOTATIONS_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/annotations/annotations_test.csv')
METADATA_CSV = Path('/home/woody/iwi5/iwi5362h/data/vindr_cxr/test_meta.csv')
LABEL_COLUMN = 'class_name'  # change only after inspecting CSV columns
BOX_COLUMNS = ['x_min', 'y_min', 'x_max', 'y_max']
LABELS = ['Cardiomegaly', 'Pleural effusion']
MAP_FIELD = {name: 'cam_vis_up' for name in SOURCE_DIRS}

# Fill with actual checkpoint paths/run identifiers. Can be a dictionary keyed
# by pathology for the separately trained guidance runs. Do not guess filenames.
CHECKPOINT_PROVENANCE = {name: '' for name in SOURCE_DIRS}
PROTOCOL_CONFIRMED = False  # set True only AFTER the inspection cell below
ALLOW_INCOMPLETE = False   # strict primary cohort; see the protocol above
SEED = 42
N_BOOT = 10000
EXPECTED_TEST_IMAGES = 3000  # official VinDr test cohort; not the gallery count
OUTPUT_PARENT = Path('./localization_pointing_game_results')
# Optional: existing PNG/JPEG used for extraction; no image is required to score.
# Example: '/path/to/png/{image_id}.png'. Raw DICOM decoding is not assumed.
IMAGE_PATH_TEMPLATE = None


## Analysis implementation
Only researcher-owned `.pt` files should be loaded: the existing serialization
uses `torch.load(..., weights_only=False)`. No models or checkpoints are executed.
The adapter accepts the explicit pathology dictionaries in the earlier notebooks,
or a root record with an explicit matching label. It refuses unidentified labels,
conflicting image IDs, duplicate filenames and ambiguous file conventions.


In [ ]:
"""Analysis of existing full-image positive heatmaps. No training or inference."""
from pathlib import Path
import hashlib
import json
import math
import platform
import sys
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy.stats import binomtest


def normal_label(value):
    return ' '.join(str(value).strip().casefold().split())


def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()


def validate_cohort(annotations, metadata, labels, label_column='class_name',
                    box_columns=('x_min','y_min','x_max','y_max')):
    """All target-positive image/finding pairs with boxes, independent of map availability."""
    for df,cols in [(annotations,['image_id',label_column,*box_columns]),
                    (metadata,['image_id','dim1','dim0'])]:
        absent=set(cols)-set(df.columns)
        if absent:raise ValueError(f'Missing columns: {sorted(absent)}')
        if df.image_id.isna().any() or df.image_id.astype(str).str.strip().eq('').any():
            raise ValueError('Missing image IDs')
    if metadata.image_id.duplicated().any():raise ValueError('Duplicate metadata image IDs')
    meta=metadata.set_index('image_id'); cohort=[]
    for label in labels:
        selected=annotations[annotations[label_column].map(normal_label)==normal_label(label)]
        if selected.empty:raise ValueError(f'No annotated positives for {label}')
        for image_id,rows in selected.groupby('image_id',sort=True):
            if image_id not in meta.index:raise ValueError(f'Missing dimensions: {image_id}')
            width,height=map(float,meta.loc[image_id,['dim1','dim0']])
            if not all(math.isfinite(v) and v>0 and v.is_integer() for v in [width,height]):
                raise ValueError(f'Invalid dim1/dim0: {image_id}')
            boxes=rows[list(box_columns)].apply(pd.to_numeric,errors='raise').drop_duplicates().to_numpy(float)
            for x0,y0,x1,y1 in boxes:
                if not (np.isfinite([x0,y0,x1,y1]).all() and 0<=x0<x1<=width and 0<=y0<y1<=height):
                    raise ValueError(f'Invalid/out-of-frame box: {image_id}/{label}: {(x0,y0,x1,y1)}')
            cohort.append(dict(image_id=str(image_id),label=label,width=int(width),height=int(height),
                               boxes=boxes.tolist(),n_boxes=len(boxes)))
    return cohort


def pointing_result(array, width, height, boxes, image_id, label, seed=42):
    """Score saved-grid global maximum, mapped by pixel centers to original coordinates.

    Ties: sample uniformly from exact maxima using a stable per-image/finding seed.
    Flat/zero maps: misses (no spatial information). No smoothing or renormalization.
    """
    a=np.asarray(array,dtype=float).squeeze()
    if a.ndim!=2 or min(a.shape)<2 or not np.isfinite(a).all() or (a<0).any():
        raise ValueError('Expected finite, nonnegative 2D positive attribution with both sides >=2')
    flat=bool(a.max()==a.min())
    base=dict(map_height=int(a.shape[0]),map_width=int(a.shape[1]),minimum=float(a.min()),
              maximum=float(a.max()),uninformative=flat)
    if flat:
        return dict(base,hit=0,peak_x=None,peak_y=None,n_maxima=int(a.size),
                    tie_expected_hit=0.,any_peak_hit=0,all_peaks_hit=0)
    yy,xx=np.where(a==a.max()); xs=(xx+.5)*width/a.shape[1]; ys=(yy+.5)*height/a.shape[0]
    inside=np.zeros(len(xx),dtype=bool)
    for x0,y0,x1,y1 in boxes:inside|=(xs>=x0)&(xs<=x1)&(ys>=y0)&(ys<=y1)
    digest=hashlib.sha256(f'{seed}|{image_id}|{normal_label(label)}'.encode()).digest()
    rng=np.random.default_rng(int.from_bytes(digest[:8],'big')); pick=int(rng.integers(len(xx)))
    return dict(base,hit=int(inside[pick]),peak_x=float(xs[pick]),peak_y=float(ys[pick]),
                n_maxima=len(xx),tie_expected_hit=float(inside.mean()),
                any_peak_hit=int(inside.any()),all_peaks_hit=int(inside.all()))


def wilson_interval(hits,n, z=1.959963984540054):
    if n==0:return (float('nan'),float('nan'))
    rate=hits/n; center=(rate+z*z/(2*n))/(1+z*z/n)
    half=z*math.sqrt(rate*(1-rate)/n+z*z/(4*n*n))/(1+z*z/n)
    return max(0.,center-half),min(1.,center+half)


def paired_statistics(baseline,candidate,n_boot=10000,seed=42):
    b=np.asarray(baseline); c=np.asarray(candidate)
    if b.ndim!=1 or b.shape!=c.shape or not len(b) or not np.isin(b,[0,1]).all() or not np.isin(c,[0,1]).all():
        raise ValueError('Paired binary outcomes required')
    d=c.astype(float)-b; rng=np.random.default_rng(seed); samples=[]
    for start in range(0,n_boot,250):
        draws=rng.integers(0,len(d),size=(min(250,n_boot-start),len(d)))
        samples.extend(d[draws].mean(axis=1))
    low,high=np.quantile(samples,[.025,.975]); wins=int(((b==0)&(c==1)).sum()); losses=int(((b==1)&(c==0)).sum())
    return dict(n=len(b),delta=float(d.mean()),delta_ci_low=float(low),delta_ci_high=float(high),
                candidate_only_hits=wins,baseline_only_hits=losses,
                p_exact=float(binomtest(wins,wins+losses,.5).pvalue) if wins+losses else 1.)


def holm_adjust(pvalues):
    p=np.asarray(pvalues,float); order=np.argsort(p); out=np.empty(len(p)); previous=0.
    for rank,index in enumerate(order):
        previous=max(previous,min(1.,(len(p)-rank)*p[index])); out[index]=previous
    return out


def load_saved(path):
    # Only load researcher-owned serialized outputs; torch pickle can execute code.
    import torch
    return torch.load(path,map_location='cpu',weights_only=False)


def extract_record(payload,image_id,label):
    if not isinstance(payload,dict):raise ValueError('Expected dictionary payload')
    meta=payload.get('__metadata__',{})
    if not isinstance(meta,dict):raise ValueError('Invalid __metadata__')
    for obj in [meta,payload]:
        if obj.get('label') is not None and normal_label(obj['label'])!=normal_label(label):
            raise ValueError('Explicit root/metadata pathology label mismatch')
        if obj.get('image_id') is not None and str(obj['image_id'])!=image_id:raise ValueError('Embedded image ID mismatch')
    matches=[key for key in payload if normal_label(key)==normal_label(label)]
    if len(matches)>1:raise ValueError('Duplicate pathology records')
    if matches:record=payload[matches[0]]
    elif normal_label(payload.get('label',meta.get('label','')))==normal_label(label):record=payload
    else:raise ValueError('No explicit matching pathology record')
    if not isinstance(record,dict):raise ValueError('Expected dictionary pathology record')
    if record.get('image_id') is not None and str(record['image_id'])!=image_id:raise ValueError('Record image ID mismatch')
    if record.get('label') is not None and normal_label(record['label'])!=normal_label(label):raise ValueError('Record label mismatch')
    return record,meta


def field_array(record,field):
    value=record
    for part in field.split('.'):value=value[part]
    if hasattr(value,'detach'):value=value.detach().cpu().numpy()
    return np.asarray(value)


def inspect_payload(value, prefix='',depth=0):
    if depth>4:return []
    if isinstance(value,dict):
        return [r for key,v in value.items() for r in inspect_payload(v,f'{prefix}.{key}' if prefix else str(key),depth+1)]
    if hasattr(value,'shape'):
        return [dict(field=prefix,shape=list(value.shape),dtype=str(value.dtype))]
    return [dict(field=prefix,value=str(value)[:250])]


def build_index(directory):
    root=Path(directory)
    if not root.is_dir():raise FileNotFoundError(root)
    index={}
    for p in sorted(root.rglob('*.pt')):
        if p.stem in index:raise ValueError(f'Duplicate file stem: {p.stem}')
        index[p.stem]=p
    if not index:raise FileNotFoundError(f'No .pt files: {root}')
    return index


def resolve_file(index,image_id,label):
    slug=normal_label(label).replace(' ','_')
    paths=[index[k] for k in [image_id,f'{image_id}__{slug}'] if k in index]
    if len(paths)>1:raise ValueError('Both image-level and per-pathology files exist; resolve ambiguity')
    if not paths:raise FileNotFoundError(f'No map for {image_id}/{label}')
    return paths[0]


def run_analysis(config,loader=load_saved):
    """Write diagnostics first; strict complete coverage is required by default."""
    if not config['protocol_confirmed']:
        raise ValueError('Inspect payloads and confirm checkpoint, attribution and full-image geometry before scoring')
    if config['baseline'] not in config['sources']:raise ValueError('Baseline not configured')
    if len(config['sources'])<2:raise ValueError('At least baseline and one comparator required')
    if config['n_boot']<100:raise ValueError('Use at least 100 bootstrap replicates (10000 recommended)')
    for method,spec in config['sources'].items():
        if not spec.get('checkpoint_provenance'):raise ValueError(f'Checkpoint provenance required: {method}')
    output=Path(config['output_dir']); output.mkdir(parents=True,exist_ok=False)
    (output/'protocol.json').write_text(json.dumps(config,indent=2,default=str))
    annotations=pd.read_csv(config['annotations_csv'],dtype={'image_id':str})
    metadata=pd.read_csv(config['metadata_csv'],dtype={'image_id':str})
    try:
        if config.get('expected_test_images') is not None and metadata.image_id.nunique()!=config['expected_test_images']:
            raise ValueError('Metadata image count differs from expected test cohort')
        cohort=validate_cohort(annotations,metadata,config['labels'],config['label_column'],tuple(config['box_columns']))
    except Exception as e:
        (output/'input_validation_error.txt').write_text(str(e)); raise
    cohort_df=pd.DataFrame(cohort); cohort_df.to_json(output/'eligible_cohort.json',orient='records',indent=2)
    rows=[]
    for method,spec in config['sources'].items():
        try:index=build_index(spec['directory']); index_error=None
        except Exception as e:index={}; index_error=str(e)
        for case in cohort:
            row={k:v for k,v in case.items() if k!='boxes'}; row['method']=method
            try:
                if index_error:raise FileNotFoundError(index_error)
                path=resolve_file(index,case['image_id'],case['label']); row['file']=str(path)
                payload=loader(path); rec,meta=extract_record(payload,case['image_id'],case['label'])
                row['file_sha256']=file_hash(path); row['embedded_metadata']=json.dumps(meta,default=str)
                row.update(pointing_result(field_array(rec,spec['map_field']),case['width'],case['height'],case['boxes'],case['image_id'],case['label'],config['seed']))
                row['status']='ok'
            except Exception as e:row.update(status='missing' if isinstance(e,FileNotFoundError) else 'invalid',error=str(e))
            rows.append(row)
    outcomes=pd.DataFrame(rows); outcomes.to_csv(output/'per_image_outcomes.csv',index=False)
    coverage=outcomes.groupby(['label','method','status']).size().rename('n').reset_index()
    coverage.to_csv(output/'coverage.csv',index=False)
    env=dict(python=sys.version,platform=platform.platform(),numpy=np.__version__,pandas=pd.__version__,
             annotations_sha256=file_hash(config['annotations_csv']),metadata_sha256=file_hash(config['metadata_csv']))
    try:
        import scipy,torch
        env.update(scipy=scipy.__version__,torch=torch.__version__)
    except ImportError:pass
    (output/'provenance.json').write_text(json.dumps(env,indent=2))
    if (outcomes.status!='ok').any() and not config['allow_incomplete']:
        raise RuntimeError(f'Missing/invalid maps: see {output}/coverage.csv. No summary calculated. Do not choose cases by outcomes.')
    summaries=[]; comparisons=[]; exclusions=[]
    for label in config['labels']:
        sub=outcomes[outcomes.label==label]
        available=sub.pivot(index='image_id',columns='method',values='status').eq('ok').all(axis=1)
        ids=available[available].index
        for image_id in available[~available].index:exclusions.append(dict(label=label,image_id=image_id,reason='not valid for every configured method'))
        if not len(ids):raise ValueError(f'No complete matched cases for {label}; diagnostics retained')
        scores=sub[sub.image_id.isin(ids)].pivot(index='image_id',columns='method',values='hit').loc[ids]
        for method in config['sources']:
            records=sub[(sub.method==method)&sub.image_id.isin(ids)]
            hits=int(scores[method].sum()); n=len(ids); low,high=wilson_interval(hits,n)
            summaries.append(dict(label=label,method=method,n_eligible=int((cohort_df.label==label).sum()),n_evaluated=n,hits=hits,hit_rate=hits/n,
                ci_low=low,ci_high=high,uninformative_maps=int(records.uninformative.sum()),
                nonflat_tied_maps=int(((records.n_maxima>1)&~records.uninformative.astype(bool)).sum()),
                tie_expected_hit_rate=float(records.tie_expected_hit.mean()),
                any_peak_hit_rate=float(records.any_peak_hit.mean()),all_peaks_hit_rate=float(records.all_peaks_hit.mean()),
                scope='exploratory complete-case intersection' if config['allow_incomplete'] else 'all eligible annotated positives'))
        for method in config['sources']:
            if method==config['baseline']:continue
            result=paired_statistics(scores[config['baseline']].to_numpy(),scores[method].to_numpy(),config['n_boot'],config['seed'])
            comparisons.append(dict(label=label,baseline=config['baseline'],method=method,**result))
    summary=pd.DataFrame(summaries); paired=pd.DataFrame(comparisons)
    paired['p_holm']=holm_adjust(paired.p_exact); paired['holm_family_size']=len(paired)
    summary.to_csv(output/'hit_rate_summary.csv',index=False); paired.to_csv(output/'paired_comparisons.csv',index=False)
    pd.DataFrame(exclusions,columns=['label','image_id','reason']).to_csv(output/'excluded_pairs.csv',index=False)
    return output,outcomes,summary,paired


## Inspect inputs before setting `PROTOCOL_CONFIRMED`
Check checkpoint metadata, image identifiers, map field/shape, target score,
extraction layer and full-image geometry. The directory name alone is not proof
of the extraction protocol. All selected models must follow the confirmed
best-validation-macro-AUC policy, including pathology-specific guidance runs.

This cell does not score any cases. If field names or payload structure differ,
modify the explicit adapter/configuration, rather than accepting an arbitrary
array. Do not use a gallery-order CSV to define the primary evaluation cohort.


In [ ]:
ann = pd.read_csv(ANNOTATIONS_CSV, dtype={'image_id': str})
meta = pd.read_csv(METADATA_CSV, dtype={'image_id': str})
print('Annotation columns:', ann.columns.tolist())
print('Dimension columns:', meta.columns.tolist(), '\nConvention: dim1=width, dim0=height')
cohort = validate_cohort(ann, meta, LABELS, LABEL_COLUMN, tuple(BOX_COLUMNS))
display(pd.DataFrame(cohort).groupby('label').agg(n_images=('image_id','nunique'), n_boxes=('n_boxes','sum')))
print('Metadata image count:', meta.image_id.nunique())
assert meta.image_id.nunique() == EXPECTED_TEST_IMAGES, 'Verify that both CSVs describe the intended test cohort'
inspection = {}
for name, directory in SOURCE_DIRS.items():
    try:
        index = build_index(directory)
        sample = next(iter(index.values()))
        payload = load_saved(sample)
        inspection[name] = {'directory': str(directory), 'n_files': len(index),
                            'sample_file': str(sample), 'fields': inspect_payload(payload)}
        print('\n', name, sample, 'files:', len(index))
        display(pd.DataFrame(inspection[name]['fields']))
    except Exception as error:
        inspection[name] = {'error': str(error)}
        print(name, type(error).__name__, str(error))
print('Now fill CHECKPOINT_PROVENANCE and confirm protocol in the configuration cell.')


## Run once with frozen choices
Re-run the configuration cell after confirming the protocol, then run this cell.
Each execution writes a new timestamped directory, so previous results are kept.
Missing/invalid coverage stops before summary statistics; inspect the path in the
exception. All-zero/constant maps remain in the denominator as misses.


In [ ]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
CONFIG = {
    'protocol_confirmed': PROTOCOL_CONFIRMED,
    'baseline': 'A0', 'labels': LABELS,
    'annotations_csv': str(ANNOTATIONS_CSV), 'metadata_csv': str(METADATA_CSV),
    'label_column': LABEL_COLUMN, 'box_columns': BOX_COLUMNS,
    'sources': {name: {'directory': str(directory), 'map_field': MAP_FIELD[name],
                       'checkpoint_provenance': CHECKPOINT_PROVENANCE[name]}
                for name, directory in SOURCE_DIRS.items()},
    'geometry': 'full_image_resize_no_crop_flip_or_padding; pixel_centers',
    'zero_flat_policy': 'miss', 'tie_policy': 'stable_seeded_uniform_exact_maximum',
    'seed': SEED, 'n_boot': N_BOOT, 'allow_incomplete': ALLOW_INCOMPLETE,
    'expected_test_images': EXPECTED_TEST_IMAGES,
    'output_dir': str(OUTPUT_PARENT / run_id),
}
output, outcomes, summary, comparisons = run_analysis(CONFIG)
(output / 'payload_inspection.json').write_text(json.dumps(inspection, indent=2))
print('Saved:', output.resolve())
display(pd.read_csv(output / 'coverage.csv'))
display(summary)
display(comparisons)


## Coverage and tie sensitivity
Check denominators before interpreting rates. `tie_expected_hit_rate` averages
hits over tied maxima; `any_peak_hit_rate` and `all_peaks_hit_rate` are diagnostic
bounds. Flat maps are misses in all of these columns. Substantial tie sensitivity
should be reported, not hidden by trying multiple seeds.


In [ ]:
display(summary[['label','method','n_eligible','n_evaluated','uninformative_maps',
                 'nonflat_tied_maps','hit_rate','tie_expected_hit_rate',
                 'all_peaks_hit_rate','any_peak_hit_rate']])
display(comparisons[['label','method','n','delta','delta_ci_low','delta_ci_high',
                     'candidate_only_hits','baseline_only_hits','p_exact','p_holm',
                     'holm_family_size']])


## Optional coordinate QC using actual saved data
Inspect the first evaluated IDs in sorted order, not a hand-picked success set.
These plots check alignment and peak coordinates; they are not additional scored
cases. Scoring uses the unmodified saved grid; the displayed heatmap uses nearest
rendering to avoid introducing a new interpolated maximum. If a radiograph path
is supplied, its dimensions must match `test_meta.csv` exactly.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

cohort_lookup = {(c['image_id'],c['label']): c for c in cohort}
for label in LABELS:
    ids = sorted(outcomes[(outcomes.label==label)&(outcomes.status=='ok')].image_id.unique())[:2]
    for image_id in ids:
        case = cohort_lookup[(image_id,label)]
        fig, axes = plt.subplots(1,len(SOURCE_DIRS),figsize=(3*len(SOURCE_DIRS),4),squeeze=False)
        for ax,(name,spec) in zip(axes[0],CONFIG['sources'].items()):
            match = outcomes[(outcomes.image_id==image_id)&(outcomes.label==label)&(outcomes.method==name)]
            if match.empty or match.iloc[0].status!='ok':
                ax.set_title(name+' — unavailable'); ax.axis('off'); continue
            row = match.iloc[0]
            record,_ = extract_record(load_saved(Path(row['file'])),image_id,label)
            array = field_array(record,spec['map_field']).squeeze()
            if IMAGE_PATH_TEMPLATE:
                im = Image.open(IMAGE_PATH_TEMPLATE.format(image_id=image_id)).convert('L')
                if im.size!=(case['width'],case['height']):
                    raise ValueError('Preview radiograph size differs from test_meta.csv')
                ax.imshow(im,cmap='gray',extent=(0,case['width'],case['height'],0))
            ax.imshow(array,cmap='magma',vmin=0,vmax=max(float(array.max()),1e-12),
                      extent=(0,case['width'],case['height'],0),interpolation='nearest',
                      alpha=.5 if IMAGE_PATH_TEMPLATE else 1.)
            for x0,y0,x1,y1 in case['boxes']:
                ax.add_patch(Rectangle((x0,y0),x1-x0,y1-y0,fill=False,color='#00A65A',lw=1.3))
            if pd.notna(row['peak_x']): ax.plot(row['peak_x'],row['peak_y'],'+',color='cyan',ms=9)
            ax.set(xlim=(0,case['width']),ylim=(case['height'],0),title=f"{name} | hit={int(row['hit'])}")
            ax.axis('off')
        fig.suptitle(f'Image ID: {image_id} | {label}')
        fig.tight_layout(); fig.savefig(output/f'coordinate_qc_{image_id}_{label.replace(" ","_")}.png',dpi=140)
        plt.show()


## Outputs to return for review

Return the **entire timestamped result folder**, especially:

- `protocol.json`, `provenance.json`, `eligible_cohort.json`, `payload_inspection.json`;
- `coverage.csv`, `excluded_pairs.csv`, `per_image_outcomes.csv`;
- `hit_rate_summary.csv`, `paired_comparisons.csv` and coordinate-QC previews.

If strict coverage fails, return the diagnostics rather than enabling incomplete
mode simply to obtain a table. If input validation fails, return
`input_validation_error.txt` and the CSV column names. If payload inspection
fails, copy its printed field/error information.

Do not update the thesis from provisional output. First review cohort coverage,
checkpoint provenance, coordinate QC, tied/flat-map rates and paired results.
No result has been generated or verified on the server by creating this notebook.
